# InteractiveLearn — Launcher

Serves all **InteractiveLearn** web apps via a local HTTP server with no-cache headers so the newest JavaScript/CSS changes load immediately.

| # | App | Page | Description |
|---|-----|------|-------------|
| 1 | **Shape Sorter** | `shapes.html` | Slow falling shapes are guided with a mouse bar into matching buckets; wrong buckets bounce shapes back |
| 2 | **Ordered Dots** | `dots.html` | A bouncing cursor ball attracts the next numbered dot; dots explode in order |
| 3 | **Toddler Reading** | `letters.html` | Floating letters form short words; type the letters for themed icon celebrations |
| 4 | **Icon Word Match** | `wordmatch.html` | See an icon and choose the matching 4–5 letter word from four choices |
| 5 | **Build the Word** | `spellword.html` | Mixed letters appear; tap them in order to fill cells while the word icon stays visible |
| 6 | **Pattern & Sound Match** | `patternsounds.html` | Choose the next shape in a pattern or match a letter sound to the right picture |
| 7 | **Number Quantity Match** | `numberquantity.html` | See a number and tap the object group with the same quantity |
| 8 | **Small to Big Sorter** | `bigsmall.html` | Tap 3–4 same objects from smallest to biggest to fill matching size cells |
| 9 | **Arrow Path Grid** | `arrowpath.html` | Follow a random arrow sequence from a beating start dot and tap the ending grid cell |
| 10 | **Hidden Ball** | `hideball.html` | Letters all same colour float around; balls hide inside them — find and select the right letter |
| 11 | **Tic-Tac-Toe** | `tictactoe.html` | Classic 2-player tic-tac-toe with click or keyboard (1-9) input |
| 12 | **Math Symbols Hunt** | `mathsymbols.html` | A bouncing cursor ball attracts target symbols; collisions explode matching symbols |
| 13 | **Egg Math** | `eggmath.html` | Object equations with eggs, oranges, carrots, and missing number keyboard answers |
| 14 | **2D Shape Surface Lab** | `measure.html` | Toddler-friendly area, perimeter, scan animations, and draggable size parameters |
| 15 | **Earth Orbit Lab** | `earthorbit.html` | Earth orbits the Sun while the Moon orbits Earth; shows day/night, moon phases, seasons, and accurate elliptical paths |
| 16 | **Gaussian Distribution Lab** | `gaussian.html` | Colored balls bounce through obstacle pegs into buckets that build a bell-shaped Gaussian distribution |
| 17 | **Earth-Centered Orbit Lab** | `geocentric.html` | Assumes Earth is fixed at the center, shows 24-hour rotation, day/night, Moon in the local sky, seasons, moon phases, and falling effects |

**How to use:**
1. Run **cell 2** once per session to start the no-cache server.
2. Run **cell 3** to open any app in the browser.
3. Re-run **cell 3** after edits; it adds a cache-busting URL version automatically.
4. Run **cell 4** to stop the server and free the port.

## Step 1 — Start the Server

Run this cell **once**. It imports everything needed, frees the port if busy, and starts the HTTP server in a background thread.

In [1]:
import os, socket, subprocess, threading, time, webbrowser
from functools import partial
from http.server import HTTPServer, SimpleHTTPRequestHandler

HOST = "localhost"
PORT = 8000
APP_DIR = os.path.dirname(os.path.abspath("__file__"))

# ── Shut down any previous server instance held in this kernel ──
_previous_httpd = globals().get("httpd")
if _previous_httpd is not None:
    _previous_httpd.shutdown()
    _previous_httpd.server_close()
    time.sleep(0.2)

# ── Release the port if another process is holding it ──
subprocess.run(["fuser", "-k", f"{PORT}/tcp"], capture_output=True, timeout=3)
time.sleep(0.25)

# ── Verify the port is actually free before binding ──
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as _s:
    _s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    if _s.connect_ex((HOST, PORT)) == 0:
        raise RuntimeError(f"Port {PORT} is still in use — close the occupying process and retry.")

# ── Create no-cache server so updated JS/CSS always loads ──
class _NoCacheHandler(SimpleHTTPRequestHandler):
    def end_headers(self):
        self.send_header("Cache-Control", "no-store, no-cache, must-revalidate, max-age=0")
        self.send_header("Pragma", "no-cache")
        self.send_header("Expires", "0")
        super().end_headers()

class _ReuseServer(HTTPServer):
    allow_reuse_address = True

handler = partial(_NoCacheHandler, directory=APP_DIR)
httpd   = _ReuseServer((HOST, PORT), handler)

server_thread = threading.Thread(target=httpd.serve_forever, daemon=True)
server_thread.start()

print(f"✔  No-cache server started  →  http://{HOST}:{PORT}")
print(f"   Serving files from: {APP_DIR}")
print("   Run the next cell to open an app with a fresh cache-busting URL.")

✔  No-cache server started  →  http://localhost:8000
   Serving files from: /home/synaps21/Downloads/InteractiveLearn
   Run the next cell to open an app with a fresh cache-busting URL.


## Step 2 — Launch an App

Set `APP_PAGE` to the desired page and run the cell. The app opens in your default browser.  
The launch URL includes `?v=<timestamp>` so updates like the new **Math Symbols** bouncing-ball collision mode, Shape Sorter, Dots, and Measure refinements load correctly without stale browser cache.

In [2]:
# ── Pick your app ─────────────────────────────────────────────
APP_PAGES = {
    "shapes":         "shapes.html",         # Shape Sorter   (slow shapes + mouse bar + matching buckets)
    "dots":           "dots.html",           # Ordered Dots   (bouncing cursor ball attracts numbered dots)
    "letters":        "letters.html",        # Toddler Reading (floating letters + typing)
    "wordmatch":      "wordmatch.html",      # Icon Word Match (choose the 4–5 letter word for the picture)
    "spellword":      "spellword.html",      # Build the Word (tap mixed letters in order while icon is visible)
    "patternsounds":  "patternsounds.html",  # Pattern & Sound Match (shape sequences + letter beginning sounds)
    "numberquantity": "numberquantity.html", # Number Quantity Match (tap the group matching the big number)
    "bigsmall":       "bigsmall.html",       # Small to Big Sorter (tap objects in size order to fill cells)
    "arrowpath":      "arrowpath.html",      # Arrow Path Grid (follow arrows from the beating dot to final cell)
    "eggmath":        "eggmath.html",        # Egg Math       (object equations + missing number answer)
    "hideball":       "hideball.html",       # Hidden Ball    (find which same-colour letter hides a ball)
    "tictactoe":      "tictactoe.html",      # Tic-Tac-Toe    (2-player board game; click or keys 1-9)
    "mathsymbols":    "mathsymbols.html",    # Math Symbols   (bouncing cursor ball + symbol collision explosions)
    "measure":        "measure.html",        # 2D Measure     (surface scans + draggable parameters)
    "earthorbit":     "earthorbit.html",     # Earth Orbit Lab (Earth-Sun-Moon paths + day/night + seasons + moon phases)
    "gaussian":       "gaussian.html",       # Gaussian Lab   (colored balls bounce through pegs into bell-curve buckets)
    "geocentric":     "geocentric.html",     # Earth-Centered Lab (Sun and Moon rotate around fixed Earth)
}

# Choose by page name or by key above.
APP_PAGE = APP_PAGES["geocentric"]
# Examples:
# APP_PAGE = APP_PAGES["geocentric"]
# APP_PAGE = APP_PAGES["gaussian"]
# APP_PAGE = APP_PAGES["earthorbit"]
# APP_PAGE = APP_PAGES["mathsymbols"]
# APP_PAGE = APP_PAGES["dots"]
# APP_PAGE = APP_PAGES["measure"]
# APP_PAGE = "spellword.html"
# ──────────────────────────────────────────────────────────────

cache_buster = int(time.time())
url = f"http://{HOST}:{PORT}/{APP_PAGE}?v={cache_buster}"
webbrowser.open(url, new=2)
print(f"🌐  Opening fresh app: {url}")
print("   If an older tab is already open, close it or refresh with Ctrl+F5.")

🌐  Opening fresh app: http://localhost:8000/geocentric.html?v=1779985752
   If an older tab is already open, close it or refresh with Ctrl+F5.


127.0.0.1 - - [28/May/2026 12:29:13] "GET /geocentric.html?v=1779985752 HTTP/1.1" 200 -


127.0.0.1 - - [28/May/2026 12:29:14] "GET /geocentric.css?v=20260507 HTTP/1.1" 200 -
127.0.0.1 - - [28/May/2026 12:29:14] "GET /js/geocentric.js?v=20260507 HTTP/1.1" 200 -
127.0.0.1 - - [28/May/2026 12:29:14] "GET /earthorbit.css?v=20260507 HTTP/1.1" 200 -
127.0.0.1 - - [28/May/2026 12:29:14] code 404, message File not found
127.0.0.1 - - [28/May/2026 12:29:14] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [28/May/2026 12:29:25] "GET /geocentric.html?v=1779985752 HTTP/1.1" 200 -
127.0.0.1 - - [28/May/2026 12:29:25] "GET /geocentric.css?v=20260507 HTTP/1.1" 200 -
127.0.0.1 - - [28/May/2026 12:29:25] "GET /js/geocentric.js?v=20260507 HTTP/1.1" 200 -
127.0.0.1 - - [28/May/2026 12:29:25] "GET /earthorbit.css?v=20260507 HTTP/1.1" 200 -
127.0.0.1 - - [28/May/2026 12:29:26] code 404, message File not found
127.0.0.1 - - [28/May/2026 12:29:26] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [28/May/2026 12:33:17] "GET /measure.html HTTP/1.1" 200 -
127.0.0.1 - - [28/May/2026 12:33:17] "GET /m

## Step 3 — Stop the Server

Run this cell to gracefully shut down the server and free port 8000.

try:

    active_httpd = globals().get("httpd")
    if active_httpd is None:
        raise RuntimeError("No active server object found in this kernel.")
    active_httpd.shutdown()try:try:try:



    active_httpd.server_close()
    print(f"✔  Server stopped — port {PORT} released.")
except Exception as e:
    print(f"Graceful shutdown failed ({e}), forcing port release ...")
    subprocess.run(["fuser", "-k", f"{PORT}/tcp"], capture_otry:
utput=True, timeout=3)
    print(f"✔  Port {PORT} force-released.")